# Chronos-Bolt Tiny — retraining da zero su dataset originale Chronos (solo training)

Questo notebook fa **solo training puro**, nessuna valutazione/inference. Retraining da zero (pesi random,
non fine-tuning) di Chronos-Bolt Tiny sul **corpus ufficiale di pre-training di Chronos**:

- `autogluon/chronos_datasets` / `training_corpus_tsmixup_10m` (10M serie, augmentazione TSMixup su dati reali)

caricato in streaming (non scaricato per intero). Non usiamo `training_corpus_kernel_synth_1m`: e' un corpus
ausiliario piu' piccolo e puramente sintetico, usato nel paper Chronos solo come componente minoritaria; per un
retraining "tiny" un solo corpus, senza logica di mixing tra due stream, e' piu' semplice da gestire ed e'
comunque rappresentativo del training set originale.

**Unici parametri variabili**: `INPUT_PATCH_SIZE` (P) e `INPUT_PATCH_STRIDE` (S), definiti come costanti in testa.
Tutti gli altri parametri (context/prediction length, batch size, learning rate, ecc.) restano fissi per garantire
confronti puliti tra run diverse.

Rispetto alla versione precedente, aggiunti due elementi presi da `train.py` originale perché proteggono la
comparabilità tra run con P/S diversi (a differenza di altri elementi come `Trainer`/campionamento pesato per
lunghezza serie, che avvicinerebbero alla pipeline ufficiale ma non aiutano a isolare l'effetto di P/S, quindi
non li abbiamo aggiunti):
- **warmup + scheduler LR lineare** (`WARMUP_RATIO`, `LR_SCHEDULER_TYPE`): evita instabilità nei primi step da
  pesi random, che potrebbe confondersi con l'effetto di P/S;
- **shuffle buffer sullo stream** (`SHUFFLE_BUFFER_SIZE`): evita correlazioni tra batch consecutivi dovute
  all'ordine dei dati nello shard.

Il modello finale viene salvato in una cartella con nome che include P e S, per poterle confrontare in seguito.

## 1. Parametri — SOLO P e S vanno cambiati tra una run e l'altra

In [ ]:
from pathlib import Path

# --- Parametri variabili dell'esperimento (i SOLI da cambiare tra le run) ---
INPUT_PATCH_SIZE = 8      # P
INPUT_PATCH_STRIDE = 4    # S

# --- Tutto il resto e' fisso ---
BASE_MODEL_ID = "amazon/chronos-bolt-tiny"   # solo riferimento architetturale, pesi NON caricati

CONTEXT_LENGTH = 2048
PREDICTION_LENGTH = 64
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Dataset ufficiale di pre-training di Chronos (streaming) — un solo corpus, vedi nota sopra
HF_REPO = "autogluon/chronos_datasets"
DATASET_CONFIG = "training_corpus_tsmixup_10m"

BATCH_SIZE = 32
MAX_STEPS = 10_000
LR = 1e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP_NORM = 1.0
LR_SCHEDULER_TYPE = "linear"   # come in train.py originale
WARMUP_RATIO = 0.05            # 5% degli step, fisso
SHUFFLE_BUFFER_SIZE = 1000     # buffer per rimescolare lo stream HF
LOG_EVERY = 50
SAVE_EVERY = 1000
SEED = 42

OUTPUT_DIR = Path(f"outputs/chronos-bolt-tiny-retrain-p{INPUT_PATCH_SIZE}-s{INPUT_PATCH_STRIDE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


## 2. Setup riproducibile

In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Dataset: corpus originale di pre-training Chronos (streaming)

Ogni riga del dataset HF ha lo schema `{id, timestamp, target}`, dove `target` e' la serie completa (lunghezza
variabile). Costruiamo un `IterableDataset` che:

1. scarta le serie troppo corte (< `CONTEXT_LENGTH + PREDICTION_LENGTH`);
2. estrae una finestra casuale `(context, target)` da ogni serie valida.

Nessun dato locale/sintetico dei generatori del progetto viene usato qui: solo il corpus ufficiale TSMixup.

In [ ]:
from datasets import load_dataset

train_stream = load_dataset(HF_REPO, DATASET_CONFIG, split="train", streaming=True)
train_stream = train_stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER_SIZE)
print("Stream TSMixup pronto (streaming=True, shuffle buffer =", SHUFFLE_BUFFER_SIZE, ")")


In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader

TOTAL_LENGTH = CONTEXT_LENGTH + PREDICTION_LENGTH

class ChronosStreamingWindowDataset(IterableDataset):
    """
    Legge lo stream HF (TSMixup) e restituisce finestre (context, mask, target, target_mask)
    pronte per ChronosBoltModelForForecasting.forward(...). Le serie troppo corte per
    context_length + prediction_length vengono scartate.
    """

    def __init__(self, hf_dataset, total_length, context_length, seed):
        self.hf_dataset = hf_dataset
        self.total_length = total_length
        self.context_length = context_length
        self.seed = seed

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        while True:
            for row in self.hf_dataset:
                values = np.asarray(row["target"], dtype=np.float32)
                if values.shape[0] < self.total_length:
                    continue
                start = int(rng.integers(0, values.shape[0] - self.total_length + 1))
                window = values[start:start + self.total_length]
                if np.isnan(window).all():
                    continue

                context = torch.tensor(window[:self.context_length], dtype=torch.float32)
                target = torch.tensor(window[self.context_length:], dtype=torch.float32)
                yield {
                    "context": context,
                    "mask": ~torch.isnan(context),
                    "target": target,
                    "target_mask": ~torch.isnan(target),
                }
            # un giro completo sul corpus e' terminato: si ricomincia (utile solo se
            # il corpus e' piu' piccolo di MAX_STEPS * BATCH_SIZE)

train_dataset = ChronosStreamingWindowDataset(
    train_stream, total_length=TOTAL_LENGTH, context_length=CONTEXT_LENGTH, seed=SEED,
)

loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=0)


## 4. Config e istanza del modello (pesi random — retraining da zero)

Scarichiamo solo la configurazione architetturale di Chronos-Bolt tiny e impostiamo `input_patch_size`/
`input_patch_stride` sui valori P/S definiti in testa. `context_length`, `prediction_length` e `quantiles`
restano quelli fissi del paper Chronos originale.

In [ ]:
from transformers import AutoConfig
from chronos.chronos_bolt import ChronosBoltModelForForecasting

config = AutoConfig.from_pretrained(BASE_MODEL_ID)

config.chronos_config["context_length"] = CONTEXT_LENGTH
config.chronos_config["prediction_length"] = PREDICTION_LENGTH
config.chronos_config["input_patch_size"] = INPUT_PATCH_SIZE
config.chronos_config["input_patch_stride"] = INPUT_PATCH_STRIDE
config.chronos_config["quantiles"] = QUANTILES

model = ChronosBoltModelForForecasting(config)  # pesi random: retraining/pretraining, non fine-tuning
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametri: {n_params/1e6:.2f}M")
print(config.chronos_config)


## 5. Training loop (solo training, nessuna valutazione)

In [ ]:
from tqdm.auto import tqdm
from transformers import get_scheduler

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
lr_scheduler = get_scheduler(
    LR_SCHEDULER_TYPE,
    optimizer=optimizer,
    num_warmup_steps=round(WARMUP_RATIO * MAX_STEPS),
    num_training_steps=MAX_STEPS,
)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

model.train()
loss_history = []
train_iter = iter(loader)

pbar = tqdm(range(1, MAX_STEPS + 1))
for step in pbar:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(loader)
        batch = next(train_iter)

    batch = {k: v.to(device) for k, v in batch.items()}

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast("cuda", enabled=use_amp):
        out = model(
            context=batch["context"],
            mask=batch["mask"],
            target=batch["target"],
            target_mask=batch["target_mask"],
        )
        loss = out.loss

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    scaler.step(optimizer)
    scaler.update()
    lr_scheduler.step()

    loss_value = float(loss.detach().cpu())
    loss_history.append(loss_value)
    pbar.set_description(f"P={INPUT_PATCH_SIZE} S={INPUT_PATCH_STRIDE} loss={loss_value:.4f} lr={lr_scheduler.get_last_lr()[0]:.2e}")

    if step % LOG_EVERY == 0:
        print(f"step={step} loss={np.mean(loss_history[-LOG_EVERY:]):.4f} lr={lr_scheduler.get_last_lr()[0]:.2e}")

    if step % SAVE_EVERY == 0:
        ckpt_dir = OUTPUT_DIR / f"checkpoint-{step}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt_dir)
        print("Salvato checkpoint:", ckpt_dir)

model.save_pretrained(OUTPUT_DIR)
print("Modello finale salvato in:", OUTPUT_DIR)


## 6. Curva della training loss (diagnostica del training, non valutazione del modello)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("step")
plt.ylabel("training loss")
plt.title(f"Chronos-Bolt retraining loss (P={INPUT_PATCH_SIZE}, S={INPUT_PATCH_STRIDE})")
plt.savefig(OUTPUT_DIR / "loss_curve.png")
plt.show()
